In [1]:
import pyro
import pyro.distributions as dist

from pyro.nn import PyroSample
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO
from pgmpy.parameter._base import BaseParameter
from skpro.distributions.normal import Normal as SkproNormal
from torch import nn
from pyro.nn import PyroModule
from pyro.nn import PyroSample

from pyro.infer import Predictive
import os
from functools import partial
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pyro.set_rng_seed(1)

%matplotlib inline
plt.style.use('default')


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class _BayesianLinearRegression(PyroModule):

    def __init__(self, in_features, mu=0., sigma=1.):
        super().__init__()
        self.linear = PyroModule[nn.Linear](in_features, 1)
        self.linear.weight = PyroSample(dist.Normal(mu, sigma).expand([1, in_features]).to_event(2))
        self.linear.bias = PyroSample(dist.Normal(mu, sigma).expand([1]).to_event(1))
        
    def forward(self, x, y=None):
        sigma = pyro.sample("sigma", dist.Uniform(0., 10.))
        mean = self.linear(x).squeeze(-1)
        with pyro.plate("data", x.shape[0]):
            obs = pyro.sample("obs", dist.Normal(mean, sigma), obs=y)
        return mean

class BayesianLinearRegression(BaseParameter):
    def __init__(
        self,
        in_features,
        num_iterations=1500,
        lr=0.03,
        posterior_samples=2000,
    ):
        super().__init__()

        self.in_features = in_features
        self.num_iterations = num_iterations
        self.lr = lr
        self.posterior_samples = posterior_samples

        self.model = None
        self.guide = None
        self._is_fitted = False

    def fit(self, X, y):
        X_tensor, y_tensor = X, y # pd.DataFrame -> tensor
        pyro.clear_param_store()
        self.model = _BayesianLinearRegression(self.in_features)
        self.guide = AutoDiagonalNormal(self.model)

        optimizer = pyro.optim.Adam({"lr": self.lr})
        svi = SVI(
            self.model,
            self.guide,
            optimizer,
            loss=Trace_ELBO(),
        )

        for j in range(self.num_iterations):
            loss = svi.step(X_tensor, y_tensor)

            if j % 100 == 0:
                print(
                    f"[iteration {j + 1:04d}] "
                    f"loss: {loss / X_tensor.shape[0]:.4f}"
                )

        self.guide.requires_grad_(False)
        self._is_fitted = True
        return self

    def predict_proba(self, X, num_samples=2000):
        X_array = X.to_numpy(dtype=float)
        index = X.index

        guide_param = next(self.guide.parameters())
        X_tensor = torch.as_tensor(
            X_array,
            dtype=guide_param.dtype,
            device=guide_param.device,
        )

        predictive = Predictive(
            model=self.model,
            guide=self.guide,
            num_samples=num_samples,
            return_sites=("obs",),
            parallel=False,
        )

        with torch.no_grad():
            samples = predictive(X_tensor)
            y_samples = samples["obs"]

        y_samples = y_samples.reshape(num_samples, X_array.shape[0])

        pred_mean = y_samples.mean(dim=0)
        pred_sigma = y_samples.std(dim=0, unbiased=False)

        eps = torch.finfo(pred_sigma.dtype).eps
        pred_sigma = pred_sigma.clamp_min(eps)

        pred_mean = pred_mean.detach().cpu().numpy()
        pred_sigma = pred_sigma.detach().cpu().numpy()

        return SkproNormal(
            mu=pred_mean.reshape(-1, 1),
            sigma=pred_sigma.reshape(-1, 1),
            index=index,
            columns=pd.Index(["y"]),
        )


In [9]:
DATA_URL = "https://github.com/pyro-ppl/datasets/blob/master/rugged_data.csv?raw=true"
data = pd.read_csv(DATA_URL, encoding="ISO-8859-1")
df = data[["cont_africa", "rugged", "rgdppc_2000"]]
df = df[np.isfinite(df.rgdppc_2000)]
df["rgdppc_2000"] = np.log(df["rgdppc_2000"])

# Dataset: Add a feature to capture the interaction between "cont_africa" and "rugged"
df["cont_africa_x_rugged"] = df["cont_africa"] * df["rugged"]
data = torch.tensor(df[["cont_africa", "rugged", "cont_africa_x_rugged", "rgdppc_2000"]].values,
                        dtype=torch.float)
x_data, y_data = data[:, :-1], data[:, -1]


In [10]:
linear_model = BayesianLinearRegression(3)


In [11]:
linear_model.fit(x_data, y_data)


[iteration 0001] loss: 3.7267
[iteration 0101] loss: 2.8423
[iteration 0201] loss: 2.2835
[iteration 0301] loss: 1.7215
[iteration 0401] loss: 1.6995
[iteration 0501] loss: 1.6810
[iteration 0601] loss: 1.7325
[iteration 0701] loss: 1.6970
[iteration 0801] loss: 1.7183
[iteration 0901] loss: 1.6994
[iteration 1001] loss: 1.6838
[iteration 1101] loss: 1.6824
[iteration 1201] loss: 1.7377
[iteration 1301] loss: 1.6912
[iteration 1401] loss: 1.6850


BayesianLinearRegression(in_features=3)

In [12]:
import pandas as pd

X_df = pd.DataFrame(
    x_data[:5].detach().cpu().numpy(),
    columns=["x1", "x2", "x3"],
)

pred_dist = linear_model.predict_proba(X_df)


In [13]:
pred_dist


Normal(columns=Index(['y'], dtype='object'),
       index=RangeIndex(start=0, stop=5, step=1),
       mu=array([[7.5227656],
       [8.683456 ],
       [8.934507 ],
       [8.9305725],
       [8.738907 ]], dtype=float32),
       sigma=array([[0.93155503],
       [0.93922466],
       [0.92554396],
       [0.9205074 ],
       [0.9447842 ]], dtype=float32))

In [8]:
pred_dist.mu.shape


(5, 1)